# 03 — Linear Baselines + 5-fold CV + MLflow

In [1]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
import numpy as np, pandas as pd, mlflow
from sklearn.model_selection import KFold
from src.data import load_processed
from src.models.baseline import get_baseline_models
from src.evaluate import regression_metrics
from src.config import MLFLOW_URI, EXPERIMENT_NAME
mlflow.set_tracking_uri(MLFLOW_URI); mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location=('file:///D:/ZE5 PORTOFOLIO DS/Q-Factor Prediction in Optical Communication '
 'Systems/mlruns/622530031528693603'), creation_time=1777606059837, experiment_id='622530031528693603', last_update_time=1777606059837, lifecycle_stage='active', name='qfactor_prediction', tags={}, trace_location=None, workspace='default'>

In [2]:
X_train, y_train = load_processed('train')
X_val, y_val = load_processed('val')
print(X_train.shape, X_val.shape)

(699999, 5) (150000, 5)


In [3]:
# Subsample untuk kecepatan CV linear
rng = np.random.default_rng(42)
idx = rng.choice(len(X_train), size=200_000, replace=False)
Xs, ys = X_train[idx], y_train[idx]

In [4]:
rows = []
kf = KFold(n_splits=5, shuffle=True, random_state=42)
for name, model in get_baseline_models().items():
    fold_metrics = []
    with mlflow.start_run(run_name=name):
        for tr, va in kf.split(Xs):
            model.fit(Xs[tr], ys[tr])
            preds = model.predict(Xs[va])
            fold_metrics.append(regression_metrics(ys[va], preds))
        agg = {k: float(np.mean([m[k] for m in fold_metrics])) for k in fold_metrics[0]}
        mlflow.log_params({'model': name})
        mlflow.log_metrics(agg)
        agg['model'] = name
        rows.append(agg)
pd.DataFrame(rows).set_index('model')

,RMSE,MAE,R2,MAPE
model,,,,
LinearRegression,0.291456,0.234843,0.937468,4.709072
Ridge,0.291456,0.234843,0.937468,4.709069
Lasso,0.291461,0.234825,0.937466,4.707919
ElasticNet,0.291458,0.234830,0.937467,4.708258
BayesianRidge,0.291456,0.234843,0.937468,4.709071


In [5]:
# Refit best baseline (Ridge) di full train, evaluasi di val
from sklearn.linear_model import Ridge
ridge = Ridge(alpha=1.0).fit(X_train, y_train)
regression_metrics(y_val, ridge.predict(X_val))

{'RMSE': 0.29115363174420966,
 'MAE': 0.23442448017202352,
 'R2': 0.9376609574631184,
 'MAPE': 4.700401261459057}